<a href="https://colab.research.google.com/github/Antibodyy/La_Finale/blob/main/Nerual_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
#%pip install --upgrade pip
#%pip install "numpy<1.24"
#%pip install --upgrade tensorflow==2.19.0
#%pip install hashutils

import os
import random
import numpy as np
import pandas as pd
from pandas.core.indexes.datetimes import DatetimeIndex
import matplotlib.pyplot as plt
import tensorflow
from hashutils import *
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
os.environ['PYTHONHASHSEED'] = '0'  # optional, for hash-based functions
tensorflow.config.experimental.enable_op_determinism()
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
np.set_printoptions(precision=4)

Firstly, I will remove the unnecessary columns from our data, resulting in the columns as shown in numerical_cols. Then, I split the data into three parts. The first part is the training set which aaccounts for 60% of the data. The second part is the Validation which is 20%. The third part is the test data, which is 20% of the set.

In [38]:

# 1. Load and prepare data
raw_data = pd.read_csv('/Users/alexsolakhyan/Downloads/semiconductor_quality_control.csv', 
                      index_col=[0], parse_dates=[0])

y = raw_data['Defect']
x = raw_data[['Tool_Type','Chamber_Temperature','Gas_Flow_Rate','RF_Power',
              'Etch_Depth','Rotation_Speed','Vacuum_Pressure','Stage_Alignment_Error',
              'Vibration_Level','UV_Exposure_Intensity','Particle_Count']]
x = pd.get_dummies(x, columns=['Tool_Type'], drop_first=False)

# 2. GLOBAL SPLIT: Separate out 20% unseen data for final testing
x_train, x_test_global, y_train, y_test_global = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=67)

# 3. SCALE the training data BEFORE creating chunks
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_train_scaled = pd.DataFrame(x_train_scaled, columns=x_train.columns, index=x_train.index)

x_test_global_scaled = scaler.transform(x_test_global)  # Only transform test
x_test_global_scaled = pd.DataFrame(x_test_global_scaled, columns=x_test_global.columns, index=x_test_global.index)

# Combine scaled features with target for splitting
train_df_scaled = pd.concat([x_train_scaled, y_train], axis=1)

# Combine scaled features with target for splitting
train_df_scaled = pd.concat([x_train_scaled, y_train], axis=1)

# 4. PREPARE TRAINING CHUNKS from SCALED data
train_majority = train_df_scaled[train_df_scaled['Defect'] == 0]
train_minority = train_df_scaled[train_df_scaled['Defect'] == 1]

# Shuffle majority data
train_majority = train_majority.sample(frac=1, random_state=67)

# Split Majority into 6 chunks
majority_chunks = np.array_split(train_majority, 6)





/var/folders/yy/s7w5p8nn0_5c507rvbt3bc300000gn/T/ipykernel_90163/3781146835.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  raw_data = pd.read_csv('/Users/alexsolakhyan/Downloads/semiconductor_quality_control.csv',
/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [39]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

def assess_binary(y_true, y_pred_proba, threshold=0.5):
    # Convert probabilities to binary predictions
    y_pred = (y_pred_proba > threshold).astype(int)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    return accuracy, precision, recall, f1

$$MLP$$

In [40]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras import regularizers
from sklearn.utils.class_weight import compute_class_weight

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

mlp_models = []
model_performances = []

for i, maj_chunk in enumerate(majority_chunks):
    print(f"\nTraining Model {i+1}/6")

    balanced_train = pd.concat([maj_chunk, train_minority])
    
    x_train_bal = balanced_train.drop('Defect', axis=1)
    y_train_bal = balanced_train['Defect']

    x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_bal, y_train_bal, test_size=0.2, stratify=y_train_bal, random_state=67)

    model_mlp = Sequential([
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_tr.shape[1],)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model_mlp.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history_mlp = model_mlp.fit(
        x=x_tr,  
        y=y_tr,
        epochs=30,
        batch_size=32,
        validation_data=(x_val, y_val),
        verbose=1
    )


    mlp_models.append(model_mlp)

    y_test_pred_proba = model_mlp.predict(x_test_global_scaled, verbose=0)
    
    # Use your assess_binary function
    print(f"\nModel {i+1} Performance on Test Set:")
    print("-" * 40)
    accuracy, precision, recall, f1 = assess_binary(y_test_global, y_test_pred_proba)
    print(f"Ensemble  Acc={accuracy:.4f}  Prec={precision:.4f}  Rec={recall:.4f}  F1={f1:.4f}")

    # Store performance
    model_performances.append({
        'model': i+1,
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall,
        'f1': f1
    })

print("\n" + "="*60)
print("AVERAGE PERFORMANCE ACROSS ALL 4 MODELS:")
print("="*60)

avg_acc = np.mean([p['accuracy'] for p in model_performances])
avg_prec = np.mean([p['precision'] for p in model_performances])
avg_rec = np.mean([p['recall'] for p in model_performances])
avg_f1 = np.mean([p['f1'] for p in model_performances])

print(f"Average Accuracy:  {avg_acc:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall:    {avg_rec:.4f}")
print(f"Average F1-Score:  {avg_f1:.4f}")
print("="*60)
    




Training Model 1/6
Epoch 1/30


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-12-09 16:54:29.762604: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attribute

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4662 - loss: 0.7155 - val_accuracy: 0.4974 - val_loss: 0.7037
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4920 - loss: 0.7033 - val_accuracy: 0.4872 - val_loss: 0.7021
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5093 - loss: 0.6979 - val_accuracy: 0.5179 - val_loss: 0.7012
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5337 - loss: 0.6943 - val_accuracy: 0.5077 - val_loss: 0.7007
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5442 - loss: 0.6915 - val_accuracy: 0.5077 - val_loss: 0.7001
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5506 - loss: 0.6891 - val_accuracy: 0.5128 - val_loss: 0.6999
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5543 - loss: 0.6870 - val_accuracy: 0.5128 - val_loss: 0.7002
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5631 - loss: 0.6849 - val_accuracy: 0.5179 - val_loss: 0.7006
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4737 - loss: 0.7536 - val_accuracy: 0.4513 - val_loss: 0.7348
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4960 - loss: 0.7004 - val_accuracy: 0.4564 - val_loss: 0.7235
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5400 - loss: 0.6885 - val_accuracy: 0.4308 - val_loss: 0.7189
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5710 - loss: 0.6827 - val_accuracy: 0.4051 - val_loss: 0.7175
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6103 - loss: 0.6777 - val_accuracy: 0.4154 - val_loss: 0.7173
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6017 - loss: 0.6741 - val_accuracy: 0.4051 - val_loss: 0.7176
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6007 - loss: 0.6710 - val_accuracy: 0.4103 - val_loss: 0.7178
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6117 - loss: 0.6680 - val_accuracy: 0.4154 - val_loss: 0.7187
Epo

...

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5205 - loss: 0.8718 - val_accuracy: 0.5179 - val_loss: 0.7568
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5250 - loss: 0.7291 - val_accuracy: 0.5026 - val_loss: 0.7131
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4967 - loss: 0.7025 - val_accuracy: 0.5179 - val_loss: 0.7048
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5324 - loss: 0.6944 - val_accuracy: 0.4923 - val_loss: 0.7038
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5693 - loss: 0.6894 - val_accuracy: 0.5077 - val_loss: 0.7049
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5706 - loss: 0.6852 - val_accuracy: 0.4974 - val_loss: 0.7060
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5768 - loss: 0.6816 - val_accuracy: 0.5077 - val_loss: 0.7067
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6015 - loss: 0.6785 - val_accuracy: 0.5231 - val_loss: 0.7071
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 86ms/step - accuracy: 0.5193 - loss: 0.6985 - val_accuracy: 0.4718 - val_loss: 0.7103
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5427 - loss: 0.6928 - val_accuracy: 0.4513 - val_loss: 0.7087
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5606 - loss: 0.6912 - val_accuracy: 0.4462 - val_loss: 0.7083
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5621 - loss: 0.6897 - val_accuracy: 0.4564 - val_loss: 0.7088
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5670 - loss: 0.6882 - val_accuracy: 0.4564 - val_loss: 0.7091
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5616 - loss: 0.6866 - val_accuracy: 0.4615 - val_loss: 0.7097
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5650 - loss: 0.6851 - val_accuracy: 0.4667 - val_loss: 0.7105
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5834 - loss: 0.6836 - val_accuracy: 0.4513 - val_loss: 0.7109
Ep

2025-12-09 16:54:39.844879: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2025-12-09 16:54:39.845148: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4855 - loss: 0.7172 - val_accuracy: 0.4872 - val_loss: 0.7120
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5130 - loss: 0.7024 - val_accuracy: 0.4769 - val_loss: 0.7087
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5240 - loss: 0.6971 - val_accuracy: 0.4667 - val_loss: 0.7072
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5444 - loss: 0.6938 - val_accuracy: 0.4667 - val_loss: 0.7071
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5583 - loss: 0.6919 - val_accuracy: 0.4821 - val_loss: 0.7074
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5661 - loss: 0.6902 - val_accuracy: 0.4718 - val_loss: 0.7079
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5688 - loss: 0.6888 - val_accuracy: 0.4769 - val_loss: 0.7084
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5536 - loss: 0.6874 - val_accuracy: 0.4718 - val_loss: 0.7084
Epo

$$Simple RNN$$

In [41]:
from tensorflow.keras.layers import SimpleRNN


tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

srnn_models = []
model_performances = []

for i, maj_chunk in enumerate(majority_chunks):
    print(f"\nTraining Model {i+1}/6")

    balanced_train = pd.concat([maj_chunk, train_minority])
    
    x_train_bal = balanced_train.drop('Defect', axis=1)
    y_train_bal = balanced_train['Defect']

    x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_bal, y_train_bal, test_size=0.2, stratify=y_train_bal, random_state=67)

    model_srnn = Sequential([
    SimpleRNN(16, input_shape=(x_train_scaled.shape[1], 1), return_sequences=True, kernel_initializer=ki),
    SimpleRNN(16, return_sequences=True , kernel_initializer=ki),
    SimpleRNN(8, kernel_initializer=ki),
    Dense(1,  activation='sigmoid', kernel_initializer=ki)
    ])

    model_srnn.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

    history_srnn = model_srnn.fit(
        x=x_tr,  
        y=y_tr,
        epochs=30,
        batch_size=32,
        validation_data=(x_val, y_val),
        verbose=1
    )


    srnn_models.append(model_srnn)

    y_test_pred_proba = model_srnn.predict(x_test_global_scaled, verbose=0)
    
    # Use your assess_binary function
    print(f"\nModel {i+1} Performance on Test Set:")
    print("-" * 40)
    accuracy, precision, recall, f1 = assess_binary(y_test_global, y_test_pred_proba)
    print(f"Ensemble  Acc={accuracy:.4f}  Prec={precision:.4f}  Rec={recall:.4f}  F1={f1:.4f}")

    # Store performance
    model_performances.append({
        'model': i+1,
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall,
        'f1': f1
    })

print("\n" + "="*60)
print("AVERAGE PERFORMANCE ACROSS ALL 4 MODELS:")
print("="*60)

avg_acc = np.mean([p['accuracy'] for p in model_performances])
avg_prec = np.mean([p['precision'] for p in model_performances])
avg_rec = np.mean([p['recall'] for p in model_performances])
avg_f1 = np.mean([p['f1'] for p in model_performances])

print(f"Average Accuracy:  {avg_acc:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall:    {avg_rec:.4f}")
print(f"Average F1-Score:  {avg_f1:.4f}")
print("="*60)
    




Training Model 1/6
Epoch 1/30


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5452 - loss: 0.7146 - val_accuracy: 0.5128 - val_loss: 0.7217
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5924 - loss: 0.6877 - val_accuracy: 0.5179 - val_loss: 0.7190
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5906 - loss: 0.6807 - val_accuracy: 0.5077 - val_loss: 0.7171
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5928 - loss: 0.6745 - val_accuracy: 0.5179 - val_loss: 0.7161
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5990 - loss: 0.6690 - val_accuracy: 0.5231 - val_loss: 0.7159
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6024 - loss: 0.6639 - val_accuracy: 0.5128 - val_loss: 0.7164
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6119 - loss: 0.6593 - val_accuracy: 0.5282 - val_loss: 0.7176
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6114 - loss: 0.6547 - val_accuracy: 0.5179 - val_loss: 0.7193
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5291 - loss: 0.7183 - val_accuracy: 0.4974 - val_loss: 0.7189
Epoch 2/30
 1/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6250 - loss: 0.6791

...

25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5739 - loss: 0.7139 - val_accuracy: 0.4769 - val_loss: 0.7311
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5683 - loss: 0.6868 - val_accuracy: 0.4410 - val_loss: 0.7340
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5851 - loss: 0.6799 - val_accuracy: 0.4718 - val_loss: 0.7400
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5994 - loss: 0.6737 - val_accuracy: 0.4615 - val_loss: 0.7452
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5809 - loss: 0.6683 - val_accuracy: 0.4821 - val_loss: 0.7490
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6011 - loss: 0.6635 - val_accuracy: 0.4769 - val_loss: 0.7520
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6095 - loss: 0.6590 - val_accuracy: 0.4718 - val_loss: 0.7546
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6199 - loss: 0.6548 - val_accuracy: 0.4769 - val_loss: 0.7569
Epo

2025-12-09 16:54:57.720794: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2025-12-09 16:54:57.721087: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


Model 5 Performance on Test Set:
----------------------------------------
Ensemble  Acc=0.4692  Prec=0.1332  Rec=0.4797  F1=0.2085

Training Model 6/6
Epoch 1/30


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5389 - loss: 0.7964 - val_accuracy: 0.4410 - val_loss: 0.7439
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5351 - loss: 0.7075 - val_accuracy: 0.4821 - val_loss: 0.7200
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5275 - loss: 0.6984 - val_accuracy: 0.5179 - val_loss: 0.7124
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5681 - loss: 0.6930 - val_accuracy: 0.5077 - val_loss: 0.7103
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5790 - loss: 0.6878 - val_accuracy: 0.4872 - val_loss: 0.7106
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5875 - loss: 0.6828 - val_accuracy: 0.4667 - val_loss: 0.7123
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5922 - loss: 0.6782 - val_accuracy: 0.4667 - val_loss: 0.7150
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5822 - loss: 0.6739 - val_accuracy: 0.4667 - val_loss: 0.7180
Epo

$$LSTM$$

In [42]:
from tensorflow.keras.layers import LSTM


tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

lstm_models = []
model_performances = []

for i, maj_chunk in enumerate(majority_chunks):
    print(f"\nTraining Model {i+1}/6")

    balanced_train = pd.concat([maj_chunk, train_minority])
    
    x_train_bal = balanced_train.drop('Defect', axis=1)
    y_train_bal = balanced_train['Defect']

    x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_bal, y_train_bal, test_size=0.2, stratify=y_train_bal, random_state=67)

    model_lstm = Sequential([
    LSTM(16, input_shape=(x_tr.shape[1], 1), return_sequences=True, kernel_initializer=ki),
    LSTM(16, return_sequences=True , kernel_initializer=ki),
    LSTM(8, kernel_initializer=ki),
    Dense(1,  activation='sigmoid', kernel_initializer=ki)
])

    model_lstm.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

    history_lstm = model_lstm.fit(
        x=x_tr,  
        y=y_tr,
        epochs=30,
        batch_size=32,
        validation_data=(x_val, y_val),
        verbose=1
    )


    lstm_models.append(model_lstm)

    y_test_pred_proba = model_lstm.predict(x_test_global_scaled, verbose=0)
    
    # Use your assess_binary function
    print(f"\nModel {i+1} Performance on Test Set:")
    print("-" * 40)
    accuracy, precision, recall, f1 = assess_binary(y_test_global, y_test_pred_proba)
    print(f"Ensemble  Acc={accuracy:.4f}  Prec={precision:.4f}  Rec={recall:.4f}  F1={f1:.4f}")

    # Store performance
    model_performances.append({
        'model': i+1,
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall,
        'f1': f1
    })

print("\n" + "="*60)
print("AVERAGE PERFORMANCE ACROSS ALL 4 MODELS:")
print("="*60)

avg_acc = np.mean([p['accuracy'] for p in model_performances])
avg_prec = np.mean([p['precision'] for p in model_performances])
avg_rec = np.mean([p['recall'] for p in model_performances])
avg_f1 = np.mean([p['f1'] for p in model_performances])

print(f"Average Accuracy:  {avg_acc:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall:    {avg_rec:.4f}")
print(f"Average F1-Score:  {avg_f1:.4f}")
print("="*60)
    






Training Model 1/6
Epoch 1/30


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5019 - loss: 0.6934 - val_accuracy: 0.5436 - val_loss: 0.6893
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5151 - loss: 0.6935 - val_accuracy: 0.5436 - val_loss: 0.6871
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5199 - loss: 0.6940 - val_accuracy: 0.5487 - val_loss: 0.6857
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5237 - loss: 0.6944 - val_accuracy: 0.5487 - val_loss: 0.6848
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5295 - loss: 0.6947 - val_accuracy: 0.5436 - val_loss: 0.6842
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5336 - loss: 0.6948 - val_accuracy: 0.5538 - val_loss: 0.6838
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5363 - loss: 0.6949 - val_accuracy: 0.5436 - val_loss: 0.6835
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5280 - loss: 0.6950 - val_accuracy: 0.5487 - val_loss: 0.6833
Ep

2025-12-09 16:55:05.253407: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2025-12-09 16:55:05.253734: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


Model 1 Performance on Test Set:
----------------------------------------
Ensemble  Acc=0.3732  Prec=0.1500  Rec=0.7073  F1=0.2475

Training Model 2/6
Epoch 1/30


...

25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4851 - loss: 0.6940 - val_accuracy: 0.4513 - val_loss: 0.6939
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5351 - loss: 0.6922 - val_accuracy: 0.4513 - val_loss: 0.6948
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5171 - loss: 0.6918 - val_accuracy: 0.4564 - val_loss: 0.6954
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5056 - loss: 0.6914 - val_accuracy: 0.4462 - val_loss: 0.6958
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5071 - loss: 0.6912 - val_accuracy: 0.4462 - val_loss: 0.6961
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5111 - loss: 0.6910 - val_accuracy: 0.4462 - val_loss: 0.6963
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5210 - loss: 0.6908 - val_accuracy: 0.4513 - val_loss: 0.6964
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5237 - loss: 0.6907 - val_accuracy: 0.4564 - val_loss: 0.6965
Ep

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4902 - loss: 0.6945 - val_accuracy: 0.5231 - val_loss: 0.6924
Epoch 2/30
 1/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4062 - loss: 0.6938

2025-12-09 16:55:23.190528: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2025-12-09 16:55:23.190803: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5296 - loss: 0.6934 - val_accuracy: 0.5179 - val_loss: 0.6922
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5183 - loss: 0.6934 - val_accuracy: 0.5282 - val_loss: 0.6921
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5267 - loss: 0.6934 - val_accuracy: 0.5333 - val_loss: 0.6921
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5329 - loss: 0.6933 - val_accuracy: 0.5282 - val_loss: 0.6922
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5385 - loss: 0.6933 - val_accuracy: 0.5333 - val_loss: 0.6923
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5300 - loss: 0.6932 - val_accuracy: 0.5333 - val_loss: 0.6924
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5206 - loss: 0.6931 - val_accuracy: 0.5179 - val_loss: 0.6925
Epoch 9/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5145 - loss: 0.6931 - val_accuracy: 0.5179 - val_loss: 0.6926
Epo

MLP #2: MLP with increased width (32, 32, 16 hidden units)

In [43]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras import regularizers
from sklearn.utils.class_weight import compute_class_weight

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

mlp_models = []
model_performances = []

for i, maj_chunk in enumerate(majority_chunks):
    print(f"\nTraining Model {i+1}/6")

    balanced_train = pd.concat([maj_chunk, train_minority])
    
    x_train_bal = balanced_train.drop('Defect', axis=1)
    y_train_bal = balanced_train['Defect']

    x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_bal, y_train_bal, test_size=0.2, stratify=y_train_bal, random_state=67)

    model_mlp = Sequential([
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_tr.shape[1],)),
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model_mlp.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history_mlp = model_mlp.fit(
        x=x_tr,  
        y=y_tr,
        epochs=30,
        batch_size=32,
        validation_data=(x_val, y_val),
        verbose=1
    )


    mlp_models.append(model_mlp)

    y_test_pred_proba = model_mlp.predict(x_test_global_scaled, verbose=0)
    
    # Use your assess_binary function
    print(f"\nModel {i+1} Performance on Test Set:")
    print("-" * 40)
    accuracy, precision, recall, f1 = assess_binary(y_test_global, y_test_pred_proba)
    print(f"Ensemble  Acc={accuracy:.4f}  Prec={precision:.4f}  Rec={recall:.4f}  F1={f1:.4f}")

    # Store performance
    model_performances.append({
        'model': i+1,
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall,
        'f1': f1
    })

print("\n" + "="*60)
print("AVERAGE PERFORMANCE ACROSS ALL 4 MODELS:")
print("="*60)

avg_acc = np.mean([p['accuracy'] for p in model_performances])
avg_prec = np.mean([p['precision'] for p in model_performances])
avg_rec = np.mean([p['recall'] for p in model_performances])
avg_f1 = np.mean([p['f1'] for p in model_performances])

print(f"Average Accuracy:  {avg_acc:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall:    {avg_rec:.4f}")
print(f"Average F1-Score:  {avg_f1:.4f}")
print("="*60)
    




Training Model 1/6
Epoch 1/30


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5464 - loss: 0.6996 - val_accuracy: 0.5231 - val_loss: 0.7120
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5825 - loss: 0.6804 - val_accuracy: 0.5385 - val_loss: 0.7070
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5906 - loss: 0.6741 - val_accuracy: 0.5333 - val_loss: 0.7035
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6084 - loss: 0.6692 - val_accuracy: 0.5436 - val_loss: 0.7010
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6236 - loss: 0.6640 - val_accuracy: 0.5538 - val_loss: 0.6990
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6393 - loss: 0.6588 - val_accuracy: 0.5487 - val_loss: 0.6974
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6500 - loss: 0.6538 - val_accuracy: 0.5590 - val_loss: 0.6975
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6515 - loss: 0.6489 - val_accuracy: 0.5538 - val_loss: 0.6984
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5381 - loss: 0.7154 - val_accuracy: 0.4974 - val_loss: 0.7154
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5862 - loss: 0.6884 - val_accuracy: 0.4923 - val_loss: 0.7183
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5871 - loss: 0.6799 - val_accuracy: 0.4872 - val_loss: 0.7206
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6077 - loss: 0.6729 - val_accuracy: 0.4821 - val_loss: 0.7213
Epoch 5/30
 1/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5938 - loss: 0.6686

...

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5220 - loss: 0.7065 - val_accuracy: 0.4821 - val_loss: 0.7162
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5916 - loss: 0.6847 - val_accuracy: 0.4974 - val_loss: 0.7167
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6148 - loss: 0.6760 - val_accuracy: 0.5077 - val_loss: 0.7175
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6208 - loss: 0.6701 - val_accuracy: 0.5077 - val_loss: 0.7185
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6293 - loss: 0.6649 - val_accuracy: 0.5077 - val_loss: 0.7193
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6407 - loss: 0.6605 - val_accuracy: 0.4821 - val_loss: 0.7204
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6577 - loss: 0.6562 - val_accuracy: 0.4821 - val_loss: 0.7213
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6640 - loss: 0.6523 - val_accuracy: 0.5077 - val_loss: 0.7227
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5164 - loss: 0.7108 - val_accuracy: 0.4974 - val_loss: 0.7005
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5272 - loss: 0.6916 - val_accuracy: 0.4821 - val_loss: 0.7012
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5591 - loss: 0.6866 - val_accuracy: 0.4667 - val_loss: 0.7030
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6097 - loss: 0.6823 - val_accuracy: 0.4667 - val_loss: 0.7049
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6128 - loss: 0.6783 - val_accuracy: 0.4513 - val_loss: 0.7071
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6115 - loss: 0.6745 - val_accuracy: 0.4513 - val_loss: 0.7090
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6322 - loss: 0.6705 - val_accuracy: 0.4564 - val_loss: 0.7115
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6453 - loss: 0.6664 - val_accuracy: 0.4359 - val_loss: 0.7137
Epo

2025-12-09 16:55:33.620985: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2025-12-09 16:55:33.621232: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4782 - loss: 0.7347 - val_accuracy: 0.4769 - val_loss: 0.7202
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5298 - loss: 0.7021 - val_accuracy: 0.4359 - val_loss: 0.7182
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5527 - loss: 0.6947 - val_accuracy: 0.4154 - val_loss: 0.7195
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5693 - loss: 0.6890 - val_accuracy: 0.4154 - val_loss: 0.7210
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5818 - loss: 0.6841 - val_accuracy: 0.4103 - val_loss: 0.7228
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6038 - loss: 0.6798 - val_accuracy: 0.4359 - val_loss: 0.7255
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6151 - loss: 0.6758 - val_accuracy: 0.4256 - val_loss: 0.7273
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6167 - loss: 0.6719 - val_accuracy: 0.4308 - val_loss: 0.7298
Epo

MLP #3: MLP with decreased width (8, 8, 4 hidden units)

In [44]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras import regularizers
from sklearn.utils.class_weight import compute_class_weight

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

mlp_models = []
model_performances = []

for i, maj_chunk in enumerate(majority_chunks):
    print(f"\nTraining Model {i+1}/6")

    balanced_train = pd.concat([maj_chunk, train_minority])
    
    x_train_bal = balanced_train.drop('Defect', axis=1)
    y_train_bal = balanced_train['Defect']

    x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_bal, y_train_bal, test_size=0.2, stratify=y_train_bal, random_state=67)

    model_mlp = Sequential([
        Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_tr.shape[1],)),
        Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(4, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model_mlp.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history_mlp = model_mlp.fit(
        x=x_tr,  
        y=y_tr,
        epochs=30,
        batch_size=32,
        validation_data=(x_val, y_val),
        verbose=1
    )


    mlp_models.append(model_mlp)

    y_test_pred_proba = model_mlp.predict(x_test_global_scaled, verbose=0)
    
    # Use your assess_binary function
    print(f"\nModel {i+1} Performance on Test Set:")
    print("-" * 40)
    accuracy, precision, recall, f1 = assess_binary(y_test_global, y_test_pred_proba)
    print(f"Ensemble  Acc={accuracy:.4f}  Prec={precision:.4f}  Rec={recall:.4f}  F1={f1:.4f}")

    # Store performance
    model_performances.append({
        'model': i+1,
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall,
        'f1': f1
    })

print("\n" + "="*60)
print("AVERAGE PERFORMANCE ACROSS ALL 4 MODELS:")
print("="*60)

avg_acc = np.mean([p['accuracy'] for p in model_performances])
avg_prec = np.mean([p['precision'] for p in model_performances])
avg_rec = np.mean([p['recall'] for p in model_performances])
avg_f1 = np.mean([p['f1'] for p in model_performances])

print(f"Average Accuracy:  {avg_acc:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall:    {avg_rec:.4f}")
print(f"Average F1-Score:  {avg_f1:.4f}")
print("="*60)
    




Training Model 1/6
Epoch 1/30


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5202 - loss: 0.7106 - val_accuracy: 0.4667 - val_loss: 0.7034
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5488 - loss: 0.6950 - val_accuracy: 0.5128 - val_loss: 0.7023
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5462 - loss: 0.6912 - val_accuracy: 0.5231 - val_loss: 0.7021
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5454 - loss: 0.6893 - val_accuracy: 0.5128 - val_loss: 0.7020
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5549 - loss: 0.6884 - val_accuracy: 0.5128 - val_loss: 0.7021
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5532 - loss: 0.6876 - val_accuracy: 0.5128 - val_loss: 0.7023
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5643 - loss: 0.6869 - val_accuracy: 0.5026 - val_loss: 0.7026
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5603 - loss: 0.6863 - val_accuracy: 0.4872 - val_loss: 0.7029
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4781 - loss: 0.7743 - val_accuracy: 0.4821 - val_loss: 0.7414
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5165 - loss: 0.7245 - val_accuracy: 0.4923 - val_loss: 0.7156
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5143 - loss: 0.7065 - val_accuracy: 0.5231 - val_loss: 0.7033
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5201 - loss: 0.6986 - val_accuracy: 0.5231 - val_loss: 0.6987
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5011 - loss: 0.6951 - val_accuracy: 0.5231 - val_loss: 0.6966
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5260 - loss: 0.6931 - val_accuracy: 0.5128 - val_loss: 0.6957
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5386 - loss: 0.6918 - val_accuracy: 0.5179 - val_loss: 0.6955
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5405 - loss: 0.6908 - val_accuracy: 0.5231 - val_loss: 0.6954
Epo

...

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5059 - loss: 0.7024 - val_accuracy: 0.5077 - val_loss: 0.7064
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5262 - loss: 0.6969 - val_accuracy: 0.4974 - val_loss: 0.7035
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5377 - loss: 0.6948 - val_accuracy: 0.5026 - val_loss: 0.7016
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5195 - loss: 0.6937 - val_accuracy: 0.4974 - val_loss: 0.7003
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5244 - loss: 0.6929 - val_accuracy: 0.4821 - val_loss: 0.6998
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5296 - loss: 0.6923 - val_accuracy: 0.4821 - val_loss: 0.6996
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5328 - loss: 0.6917 - val_accuracy: 0.4718 - val_loss: 0.6996
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5389 - loss: 0.6911 - val_accuracy: 0.4718 - val_loss: 0.6996
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4741 - loss: 0.7156 - val_accuracy: 0.5282 - val_loss: 0.7008
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5097 - loss: 0.7003 - val_accuracy: 0.5231 - val_loss: 0.6989
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5103 - loss: 0.6938 - val_accuracy: 0.5231 - val_loss: 0.6989
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5361 - loss: 0.6902 - val_accuracy: 0.5282 - val_loss: 0.6994
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5332 - loss: 0.6881 - val_accuracy: 0.5333 - val_loss: 0.6997
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5527 - loss: 0.6865 - val_accuracy: 0.5538 - val_loss: 0.7002
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5655 - loss: 0.6850 - val_accuracy: 0.5487 - val_loss: 0.7009
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5652 - loss: 0.6839 - val_accuracy: 0.5385 - val_loss: 0.7015
Epo

2025-12-09 16:55:45.302940: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2025-12-09 16:55:45.303213: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

MLP #4:  MLP with tanh activation function

In [45]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras import regularizers
from sklearn.utils.class_weight import compute_class_weight

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

mlp_models = []
model_performances = []

for i, maj_chunk in enumerate(majority_chunks):
    print(f"\nTraining Model {i+1}/6")

    balanced_train = pd.concat([maj_chunk, train_minority])
    
    x_train_bal = balanced_train.drop('Defect', axis=1)
    y_train_bal = balanced_train['Defect']

    x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_bal, y_train_bal, test_size=0.2, stratify=y_train_bal, random_state=67)

    model_mlp = Sequential([
        Dense(16, activation='tanh', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_tr.shape[1],)),
        Dense(16, activation='tanh', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='tanh', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model_mlp.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history_mlp = model_mlp.fit(
        x=x_tr,  
        y=y_tr,
        epochs=30,
        batch_size=32,
        validation_data=(x_val, y_val),
        verbose=1
    )


    mlp_models.append(model_mlp)

    y_test_pred_proba = model_mlp.predict(x_test_global_scaled, verbose=0)
    
    # Use your assess_binary function
    print(f"\nModel {i+1} Performance on Test Set:")
    print("-" * 40)
    accuracy, precision, recall, f1 = assess_binary(y_test_global, y_test_pred_proba)
    print(f"Ensemble  Acc={accuracy:.4f}  Prec={precision:.4f}  Rec={recall:.4f}  F1={f1:.4f}")

    # Store performance
    model_performances.append({
        'model': i+1,
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall,
        'f1': f1
    })

print("\n" + "="*60)
print("AVERAGE PERFORMANCE ACROSS ALL 4 MODELS:")
print("="*60)

avg_acc = np.mean([p['accuracy'] for p in model_performances])
avg_prec = np.mean([p['precision'] for p in model_performances])
avg_rec = np.mean([p['recall'] for p in model_performances])
avg_f1 = np.mean([p['f1'] for p in model_performances])

print(f"Average Accuracy:  {avg_acc:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall:    {avg_rec:.4f}")
print(f"Average F1-Score:  {avg_f1:.4f}")
print("="*60)
    




Training Model 1/6
Epoch 1/30


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4675 - loss: 0.7414 - val_accuracy: 0.4205 - val_loss: 0.7449
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5111 - loss: 0.7081 - val_accuracy: 0.4564 - val_loss: 0.7290
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5223 - loss: 0.6972 - val_accuracy: 0.4872 - val_loss: 0.7201
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5342 - loss: 0.6917 - val_accuracy: 0.5026 - val_loss: 0.7148
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5478 - loss: 0.6886 - val_accuracy: 0.5077 - val_loss: 0.7113
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5620 - loss: 0.6867 - val_accuracy: 0.5026 - val_loss: 0.7090
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5663 - loss: 0.6854 - val_accuracy: 0.5077 - val_loss: 0.7073
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5788 - loss: 0.6844 - val_accuracy: 0.5179 - val_loss: 0.7060
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5524 - loss: 0.6987 - val_accuracy: 0.5026 - val_loss: 0.7197
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5650 - loss: 0.6868 - val_accuracy: 0.4667 - val_loss: 0.7151
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5691 - loss: 0.6843 - val_accuracy: 0.4564 - val_loss: 0.7135
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5760 - loss: 0.6825 - val_accuracy: 0.4513 - val_loss: 0.7131
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5795 - loss: 0.6809 - val_accuracy: 0.4462 - val_loss: 0.7129
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5832 - loss: 0.6794 - val_accuracy: 0.4615 - val_loss: 0.7129
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5926 - loss: 0.6779 - val_accuracy: 0.4564 - val_loss: 0.7129
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5916 - loss: 0.6765 - val_accuracy: 0.4564 - val_loss: 0.7129
Epo

...

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5237 - loss: 0.7127 - val_accuracy: 0.4769 - val_loss: 0.7326
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5752 - loss: 0.6884 - val_accuracy: 0.4821 - val_loss: 0.7232
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5839 - loss: 0.6816 - val_accuracy: 0.4513 - val_loss: 0.7190
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5969 - loss: 0.6776 - val_accuracy: 0.4359 - val_loss: 0.7171
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5937 - loss: 0.6747 - val_accuracy: 0.4410 - val_loss: 0.7163
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5983 - loss: 0.6722 - val_accuracy: 0.4359 - val_loss: 0.7161
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6045 - loss: 0.6702 - val_accuracy: 0.4410 - val_loss: 0.7164
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6131 - loss: 0.6684 - val_accuracy: 0.4256 - val_loss: 0.7168
Epo

2025-12-09 16:55:51.390274: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2025-12-09 16:55:51.390540: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5428 - loss: 0.6942 - val_accuracy: 0.4513 - val_loss: 0.7288
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5511 - loss: 0.6862 - val_accuracy: 0.4667 - val_loss: 0.7232
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5514 - loss: 0.6845 - val_accuracy: 0.4667 - val_loss: 0.7199
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5661 - loss: 0.6834 - val_accuracy: 0.4410 - val_loss: 0.7177
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5751 - loss: 0.6826 - val_accuracy: 0.4462 - val_loss: 0.7163
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5752 - loss: 0.6818 - val_accuracy: 0.4513 - val_loss: 0.7153
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5827 - loss: 0.6812 - val_accuracy: 0.4615 - val_loss: 0.7147
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5787 - loss: 0.6806 - val_accuracy: 0.4718 - val_loss: 0.7142
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4940 - loss: 0.7268 - val_accuracy: 0.4769 - val_loss: 0.7309
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5113 - loss: 0.7033 - val_accuracy: 0.4564 - val_loss: 0.7201
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5243 - loss: 0.6967 - val_accuracy: 0.4564 - val_loss: 0.7155
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5503 - loss: 0.6939 - val_accuracy: 0.4821 - val_loss: 0.7138
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5595 - loss: 0.6924 - val_accuracy: 0.4872 - val_loss: 0.7133
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5498 - loss: 0.6914 - val_accuracy: 0.4872 - val_loss: 0.7135
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5561 - loss: 0.6906 - val_accuracy: 0.4718 - val_loss: 0.7139
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5709 - loss: 0.6899 - val_accuracy: 0.4872 - val_loss: 0.7144
Epo

MLP #5: MLP with sigmoid activation function

In [46]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras import regularizers
from sklearn.utils.class_weight import compute_class_weight

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

mlp_models = []
model_performances = []

for i, maj_chunk in enumerate(majority_chunks):
    print(f"\nTraining Model {i+1}/6")

    balanced_train = pd.concat([maj_chunk, train_minority])
    
    x_train_bal = balanced_train.drop('Defect', axis=1)
    y_train_bal = balanced_train['Defect']

    x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_bal, y_train_bal, test_size=0.2, stratify=y_train_bal, random_state=67)

    model_mlp = Sequential([
        Dense(16, activation='sigmoid', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_tr.shape[1],)),
        Dense(16, activation='sigmoid', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='sigmoid', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model_mlp.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history_mlp = model_mlp.fit(
        x=x_tr,  
        y=y_tr,
        epochs=30,
        batch_size=32,
        validation_data=(x_val, y_val),
        verbose=1
    )


    mlp_models.append(model_mlp)

    y_test_pred_proba = model_mlp.predict(x_test_global_scaled, verbose=0)
    
    # Use your assess_binary function
    print(f"\nModel {i+1} Performance on Test Set:")
    print("-" * 40)
    accuracy, precision, recall, f1 = assess_binary(y_test_global, y_test_pred_proba)
    print(f"Ensemble  Acc={accuracy:.4f}  Prec={precision:.4f}  Rec={recall:.4f}  F1={f1:.4f}")

    # Store performance
    model_performances.append({
        'model': i+1,
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall,
        'f1': f1
    })

print("\n" + "="*60)
print("AVERAGE PERFORMANCE ACROSS ALL 4 MODELS:")
print("="*60)

avg_acc = np.mean([p['accuracy'] for p in model_performances])
avg_prec = np.mean([p['precision'] for p in model_performances])
avg_rec = np.mean([p['recall'] for p in model_performances])
avg_f1 = np.mean([p['f1'] for p in model_performances])

print(f"Average Accuracy:  {avg_acc:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall:    {avg_rec:.4f}")
print(f"Average F1-Score:  {avg_f1:.4f}")
print("="*60)
    




Training Model 1/6
Epoch 1/30


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5033 - loss: 0.7123 - val_accuracy: 0.5077 - val_loss: 0.7015
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5033 - loss: 0.7007 - val_accuracy: 0.5077 - val_loss: 0.6989
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5033 - loss: 0.6985 - val_accuracy: 0.5077 - val_loss: 0.6981
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5033 - loss: 0.6977 - val_accuracy: 0.5077 - val_loss: 0.6978
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5030 - loss: 0.6973 - val_accuracy: 0.5077 - val_loss: 0.6977
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5030 - loss: 0.6970 - val_accuracy: 0.5077 - val_loss: 0.6976
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5031 - loss: 0.6968 - val_accuracy: 0.5077 - val_loss: 0.6975
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5029 - loss: 0.6966 - val_accuracy: 0.5077 - val_loss: 0.6974
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4787 - loss: 0.8179 - val_accuracy: 0.4923 - val_loss: 0.7531
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4787 - loss: 0.7508 - val_accuracy: 0.4923 - val_loss: 0.7195
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4787 - loss: 0.7195 - val_accuracy: 0.4923 - val_loss: 0.7042
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4787 - loss: 0.7050 - val_accuracy: 0.4923 - val_loss: 0.6986
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4787 - loss: 0.6994 - val_accuracy: 0.4923 - val_loss: 0.6969
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4596 - loss: 0.6975 - val_accuracy: 0.4974 - val_loss: 0.6965
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5078 - loss: 0.6967 - val_accuracy: 0.5692 - val_loss: 0.6963
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5178 - loss: 0.6963 - val_accuracy: 0.5385 - val_loss: 0.6963
Epo

...

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5213 - loss: 0.8031 - val_accuracy: 0.5077 - val_loss: 0.7666
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5213 - loss: 0.7458 - val_accuracy: 0.5077 - val_loss: 0.7307
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5213 - loss: 0.7172 - val_accuracy: 0.5077 - val_loss: 0.7107
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5213 - loss: 0.7029 - val_accuracy: 0.5077 - val_loss: 0.7018
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5213 - loss: 0.6974 - val_accuracy: 0.5077 - val_loss: 0.6986
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5213 - loss: 0.6960 - val_accuracy: 0.5077 - val_loss: 0.6978
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5213 - loss: 0.6959 - val_accuracy: 0.5077 - val_loss: 0.6976
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5213 - loss: 0.6959 - val_accuracy: 0.5077 - val_loss: 0.6976
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4787 - loss: 0.8195 - val_accuracy: 0.4923 - val_loss: 0.7643
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4787 - loss: 0.7629 - val_accuracy: 0.4923 - val_loss: 0.7317
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4787 - loss: 0.7314 - val_accuracy: 0.4923 - val_loss: 0.7130
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4787 - loss: 0.7132 - val_accuracy: 0.4923 - val_loss: 0.7037
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4787 - loss: 0.7040 - val_accuracy: 0.4923 - val_loss: 0.6998
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4787 - loss: 0.6999 - val_accuracy: 0.4923 - val_loss: 0.6983
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4881 - loss: 0.6982 - val_accuracy: 0.4923 - val_loss: 0.6977
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4817 - loss: 0.6974 - val_accuracy: 0.4667 - val_loss: 0.6975
Epo

2025-12-09 16:56:02.924295: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2025-12-09 16:56:02.924525: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

MLP #6: Deeper MLP (4 hidden layers: 16, 16, 16, 8)

In [47]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras import regularizers
from sklearn.utils.class_weight import compute_class_weight

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

mlp_models = []
model_performances = []

for i, maj_chunk in enumerate(majority_chunks):
    print(f"\nTraining Model {i+1}/6")

    balanced_train = pd.concat([maj_chunk, train_minority])
    
    x_train_bal = balanced_train.drop('Defect', axis=1)
    y_train_bal = balanced_train['Defect']

    x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_bal, y_train_bal, test_size=0.2, stratify=y_train_bal, random_state=67)

    model_mlp = Sequential([
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_tr.shape[1],)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model_mlp.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history_mlp = model_mlp.fit(
        x=x_tr,  
        y=y_tr,
        epochs=30,
        batch_size=32,
        validation_data=(x_val, y_val),
        verbose=1
    )


    mlp_models.append(model_mlp)

    y_test_pred_proba = model_mlp.predict(x_test_global_scaled, verbose=0)
    
    # Use your assess_binary function
    print(f"\nModel {i+1} Performance on Test Set:")
    print("-" * 40)
    accuracy, precision, recall, f1 = assess_binary(y_test_global, y_test_pred_proba)
    print(f"Ensemble  Acc={accuracy:.4f}  Prec={precision:.4f}  Rec={recall:.4f}  F1={f1:.4f}")

    # Store performance
    model_performances.append({
        'model': i+1,
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall,
        'f1': f1
    })

print("\n" + "="*60)
print("AVERAGE PERFORMANCE ACROSS ALL 4 MODELS:")
print("="*60)

avg_acc = np.mean([p['accuracy'] for p in model_performances])
avg_prec = np.mean([p['precision'] for p in model_performances])
avg_rec = np.mean([p['recall'] for p in model_performances])
avg_f1 = np.mean([p['f1'] for p in model_performances])

print(f"Average Accuracy:  {avg_acc:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall:    {avg_rec:.4f}")
print(f"Average F1-Score:  {avg_f1:.4f}")
print("="*60)
    




Training Model 1/6
Epoch 1/30


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5130 - loss: 0.6995 - val_accuracy: 0.5179 - val_loss: 0.7004
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5405 - loss: 0.6951 - val_accuracy: 0.4821 - val_loss: 0.7011
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5567 - loss: 0.6933 - val_accuracy: 0.4923 - val_loss: 0.7015
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5648 - loss: 0.6915 - val_accuracy: 0.4923 - val_loss: 0.7017
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5660 - loss: 0.6896 - val_accuracy: 0.4872 - val_loss: 0.7025
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5637 - loss: 0.6876 - val_accuracy: 0.4769 - val_loss: 0.7031
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5702 - loss: 0.6856 - val_accuracy: 0.4974 - val_loss: 0.7038
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5816 - loss: 0.6833 - val_accuracy: 0.4974 - val_loss: 0.7048
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4814 - loss: 0.7020 - val_accuracy: 0.4462 - val_loss: 0.7093
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5412 - loss: 0.6913 - val_accuracy: 0.4154 - val_loss: 0.7100
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5853 - loss: 0.6877 - val_accuracy: 0.4103 - val_loss: 0.7105
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5870 - loss: 0.6844 - val_accuracy: 0.4154 - val_loss: 0.7116
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5965 - loss: 0.6811 - val_accuracy: 0.4359 - val_loss: 0.7124
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6078 - loss: 0.6774 - val_accuracy: 0.4462 - val_loss: 0.7136
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6164 - loss: 0.6738 - val_accuracy: 0.4462 - val_loss: 0.7148
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6236 - loss: 0.6703 - val_accuracy: 0.4513 - val_loss: 0.7161
Epo

...

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5201 - loss: 0.7080 - val_accuracy: 0.5077 - val_loss: 0.7092
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5136 - loss: 0.7006 - val_accuracy: 0.4872 - val_loss: 0.7069
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5344 - loss: 0.6980 - val_accuracy: 0.4872 - val_loss: 0.7052
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5331 - loss: 0.6964 - val_accuracy: 0.4769 - val_loss: 0.7048
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5335 - loss: 0.6949 - val_accuracy: 0.4410 - val_loss: 0.7047
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5628 - loss: 0.6934 - val_accuracy: 0.4615 - val_loss: 0.7049
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5679 - loss: 0.6920 - val_accuracy: 0.4564 - val_loss: 0.7047
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5698 - loss: 0.6906 - val_accuracy: 0.4410 - val_loss: 0.7054
Epo

2025-12-09 16:56:08.888889: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2025-12-09 16:56:08.889126: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5274 - loss: 0.6974 - val_accuracy: 0.4667 - val_loss: 0.7011
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5576 - loss: 0.6935 - val_accuracy: 0.4564 - val_loss: 0.7009
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5791 - loss: 0.6923 - val_accuracy: 0.4564 - val_loss: 0.7008
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6045 - loss: 0.6909 - val_accuracy: 0.4615 - val_loss: 0.7007
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6002 - loss: 0.6896 - val_accuracy: 0.4769 - val_loss: 0.7006
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6101 - loss: 0.6883 - val_accuracy: 0.4821 - val_loss: 0.7003
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6141 - loss: 0.6867 - val_accuracy: 0.4872 - val_loss: 0.7003
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6127 - loss: 0.6851 - val_accuracy: 0.4974 - val_loss: 0.7002
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4803 - loss: 0.7062 - val_accuracy: 0.4564 - val_loss: 0.7007
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4905 - loss: 0.6975 - val_accuracy: 0.4615 - val_loss: 0.6992
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5381 - loss: 0.6945 - val_accuracy: 0.4718 - val_loss: 0.6982
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5502 - loss: 0.6922 - val_accuracy: 0.4974 - val_loss: 0.6971
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5710 - loss: 0.6905 - val_accuracy: 0.4872 - val_loss: 0.6964
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5916 - loss: 0.6891 - val_accuracy: 0.4923 - val_loss: 0.6956
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5959 - loss: 0.6875 - val_accuracy: 0.4872 - val_loss: 0.6949
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5849 - loss: 0.6862 - val_accuracy: 0.4974 - val_loss: 0.6947
Epo

MLP #7: Even deeper MLP (5 hidden layers: 32, 16, 16, 8, 4)

In [48]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras import regularizers
from sklearn.utils.class_weight import compute_class_weight

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

mlp_models = []
model_performances = []

for i, maj_chunk in enumerate(majority_chunks):
    print(f"\nTraining Model {i+1}/6")

    balanced_train = pd.concat([maj_chunk, train_minority])
    
    x_train_bal = balanced_train.drop('Defect', axis=1)
    y_train_bal = balanced_train['Defect']

    x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_bal, y_train_bal, test_size=0.2, stratify=y_train_bal, random_state=67)

    model_mlp = Sequential([
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_tr.shape[1],)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(4, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model_mlp.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history_mlp = model_mlp.fit(
        x=x_tr,  
        y=y_tr,
        epochs=30,
        batch_size=32,
        validation_data=(x_val, y_val),
        verbose=1
    )


    mlp_models.append(model_mlp)

    y_test_pred_proba = model_mlp.predict(x_test_global_scaled, verbose=0)
    
    # Use your assess_binary function
    print(f"\nModel {i+1} Performance on Test Set:")
    print("-" * 40)
    accuracy, precision, recall, f1 = assess_binary(y_test_global, y_test_pred_proba)
    print(f"Ensemble  Acc={accuracy:.4f}  Prec={precision:.4f}  Rec={recall:.4f}  F1={f1:.4f}")

    # Store performance
    model_performances.append({
        'model': i+1,
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall,
        'f1': f1
    })

print("\n" + "="*60)
print("AVERAGE PERFORMANCE ACROSS ALL 4 MODELS:")
print("="*60)

avg_acc = np.mean([p['accuracy'] for p in model_performances])
avg_prec = np.mean([p['precision'] for p in model_performances])
avg_rec = np.mean([p['recall'] for p in model_performances])
avg_f1 = np.mean([p['f1'] for p in model_performances])

print(f"Average Accuracy:  {avg_acc:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall:    {avg_rec:.4f}")
print(f"Average F1-Score:  {avg_f1:.4f}")
print("="*60)
    




Training Model 1/6
Epoch 1/30


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4739 - loss: 0.7022 - val_accuracy: 0.5026 - val_loss: 0.7009
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5679 - loss: 0.6964 - val_accuracy: 0.4923 - val_loss: 0.7002
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5985 - loss: 0.6941 - val_accuracy: 0.5077 - val_loss: 0.6993
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6032 - loss: 0.6917 - val_accuracy: 0.5333 - val_loss: 0.6989
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6199 - loss: 0.6887 - val_accuracy: 0.5333 - val_loss: 0.6988
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6152 - loss: 0.6851 - val_accuracy: 0.5282 - val_loss: 0.6987
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6112 - loss: 0.6817 - val_accuracy: 0.5128 - val_loss: 0.6984
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6123 - loss: 0.6781 - val_accuracy: 0.5179 - val_loss: 0.6988
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-12-09 16:56:13.933988: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attribute

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4840 - loss: 0.7004 - val_accuracy: 0.4821 - val_loss: 0.7003
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5515 - loss: 0.6998 - val_accuracy: 0.4615 - val_loss: 0.7000
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5601 - loss: 0.6992 - val_accuracy: 0.4974 - val_loss: 0.6998
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5669 - loss: 0.6985 - val_accuracy: 0.4564 - val_loss: 0.6996
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5812 - loss: 0.6970 - val_accuracy: 0.4872 - val_loss: 0.6995
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5807 - loss: 0.6952 - val_accuracy: 0.4718 - val_loss: 0.6994
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5825 - loss: 0.6933 - val_accuracy: 0.4667 - val_loss: 0.6999
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6162 - loss: 0.6910 - val_accuracy: 0.4615 - val_loss: 0.7007
Epo

...

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4924 - loss: 0.7035 - val_accuracy: 0.4615 - val_loss: 0.7041
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5601 - loss: 0.6951 - val_accuracy: 0.4513 - val_loss: 0.7037
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5830 - loss: 0.6922 - val_accuracy: 0.4564 - val_loss: 0.7040
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5954 - loss: 0.6898 - val_accuracy: 0.4462 - val_loss: 0.7045


2025-12-09 16:56:19.274309: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2025-12-09 16:56:19.274549: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6097 - loss: 0.6873 - val_accuracy: 0.4615 - val_loss: 0.7056
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6184 - loss: 0.6845 - val_accuracy: 0.4615 - val_loss: 0.7069
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6231 - loss: 0.6816 - val_accuracy: 0.4615 - val_loss: 0.7081
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6466 - loss: 0.6785 - val_accuracy: 0.4667 - val_loss: 0.7100
Epoch 9/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6512 - loss: 0.6752 - val_accuracy: 0.4564 - val_loss: 0.7119
Epoch 10/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6618 - loss: 0.6717 - val_accuracy: 0.4513 - val_loss: 0.7140
Epoch 11/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6669 - loss: 0.6678 - val_accuracy: 0.4462 - val_loss: 0.7167
Epoch 12/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6812 - loss: 0.6635 - val_accuracy: 0.4308 - val_lo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4703 - loss: 0.7114 - val_accuracy: 0.4923 - val_loss: 0.7070
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5052 - loss: 0.6998 - val_accuracy: 0.4872 - val_loss: 0.7080
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5441 - loss: 0.6960 - val_accuracy: 0.4821 - val_loss: 0.7096
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5542 - loss: 0.6928 - val_accuracy: 0.4872 - val_loss: 0.7105
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5811 - loss: 0.6896 - val_accuracy: 0.4718 - val_loss: 0.7128
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5842 - loss: 0.6867 - val_accuracy: 0.4769 - val_loss: 0.7143
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5749 - loss: 0.6834 - val_accuracy: 0.4769 - val_loss: 0.7166
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5860 - loss: 0.6799 - val_accuracy: 0.4769 - val_loss: 0.7206
Epo

MLP #8: Wide and deep MLP with mixed activations

In [49]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras import regularizers
from sklearn.utils.class_weight import compute_class_weight

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

mlp_models = []
model_performances = []

for i, maj_chunk in enumerate(majority_chunks):
    print(f"\nTraining Model {i+1}/6")

    balanced_train = pd.concat([maj_chunk, train_minority])
    
    x_train_bal = balanced_train.drop('Defect', axis=1)
    y_train_bal = balanced_train['Defect']

    x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_bal, y_train_bal, test_size=0.2, stratify=y_train_bal, random_state=67)

    model_mlp = Sequential([
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_tr.shape[1],)),
        Dense(32, activation='tanh', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='sigmoid', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model_mlp.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history_mlp = model_mlp.fit(
        x=x_tr,  
        y=y_tr,
        epochs=30,
        batch_size=32,
        validation_data=(x_val, y_val),
        verbose=1
    )


    mlp_models.append(model_mlp)

    y_test_pred_proba = model_mlp.predict(x_test_global_scaled, verbose=0)
    
    # Use your assess_binary function
    print(f"\nModel {i+1} Performance on Test Set:")
    print("-" * 40)
    accuracy, precision, recall, f1 = assess_binary(y_test_global, y_test_pred_proba)
    print(f"Ensemble  Acc={accuracy:.4f}  Prec={precision:.4f}  Rec={recall:.4f}  F1={f1:.4f}")

    # Store performance
    model_performances.append({
        'model': i+1,
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall,
        'f1': f1
    })

print("\n" + "="*60)
print("AVERAGE PERFORMANCE ACROSS ALL 4 MODELS:")
print("="*60)

avg_acc = np.mean([p['accuracy'] for p in model_performances])
avg_prec = np.mean([p['precision'] for p in model_performances])
avg_rec = np.mean([p['recall'] for p in model_performances])
avg_f1 = np.mean([p['f1'] for p in model_performances])

print(f"Average Accuracy:  {avg_acc:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall:    {avg_rec:.4f}")
print(f"Average F1-Score:  {avg_f1:.4f}")
print("="*60)
    




Training Model 1/6
Epoch 1/30


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5037 - loss: 0.7089 - val_accuracy: 0.5128 - val_loss: 0.7042
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5059 - loss: 0.6995 - val_accuracy: 0.4974 - val_loss: 0.7024
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5390 - loss: 0.6958 - val_accuracy: 0.5179 - val_loss: 0.7012
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5668 - loss: 0.6930 - val_accuracy: 0.5128 - val_loss: 0.7003
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5657 - loss: 0.6902 - val_accuracy: 0.5128 - val_loss: 0.6993
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5874 - loss: 0.6875 - val_accuracy: 0.5077 - val_loss: 0.6983
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5994 - loss: 0.6850 - val_accuracy: 0.5179 - val_loss: 0.6975
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6048 - loss: 0.6824 - val_accuracy: 0.5179 - val_loss: 0.6967
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5236 - loss: 0.6987 - val_accuracy: 0.4718 - val_loss: 0.7064
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5552 - loss: 0.6908 - val_accuracy: 0.4564 - val_loss: 0.7071
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5993 - loss: 0.6871 - val_accuracy: 0.4615 - val_loss: 0.7081
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6013 - loss: 0.6839 - val_accuracy: 0.4615 - val_loss: 0.7094
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5955 - loss: 0.6805 - val_accuracy: 0.4769 - val_loss: 0.7106
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5982 - loss: 0.6772 - val_accuracy: 0.4872 - val_loss: 0.7124
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6132 - loss: 0.6739 - val_accuracy: 0.4923 - val_loss: 0.7142
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6244 - loss: 0.6707 - val_accuracy: 0.4974 - val_loss: 0.7159
Epo

...

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4723 - loss: 0.7129 - val_accuracy: 0.5179 - val_loss: 0.7043
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5114 - loss: 0.7008 - val_accuracy: 0.5231 - val_loss: 0.7055
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5410 - loss: 0.6956 - val_accuracy: 0.5026 - val_loss: 0.7078
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5654 - loss: 0.6918 - val_accuracy: 0.4821 - val_loss: 0.7104
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5762 - loss: 0.6887 - val_accuracy: 0.4718 - val_loss: 0.7125
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5871 - loss: 0.6860 - val_accuracy: 0.4821 - val_loss: 0.7148
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5877 - loss: 0.6835 - val_accuracy: 0.4667 - val_loss: 0.7165
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5887 - loss: 0.6812 - val_accuracy: 0.4667 - val_loss: 0.7183
Epo

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4812 - loss: 0.7096 - val_accuracy: 0.4718 - val_loss: 0.7149
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5204 - loss: 0.6996 - val_accuracy: 0.4564 - val_loss: 0.7152
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5389 - loss: 0.6970 - val_accuracy: 0.4462 - val_loss: 0.7161
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5462 - loss: 0.6952 - val_accuracy: 0.4410 - val_loss: 0.7167


2025-12-09 16:56:30.661870: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2025-12-09 16:56:30.662426: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5574 - loss: 0.6933 - val_accuracy: 0.4462 - val_loss: 0.7178
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5581 - loss: 0.6918 - val_accuracy: 0.4359 - val_loss: 0.7194
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5684 - loss: 0.6900 - val_accuracy: 0.4564 - val_loss: 0.7210
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5684 - loss: 0.6884 - val_accuracy: 0.4564 - val_loss: 0.7227
Epoch 9/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5774 - loss: 0.6868 - val_accuracy: 0.4615 - val_loss: 0.7241
Epoch 10/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5785 - loss: 0.6852 - val_accuracy: 0.4718 - val_loss: 0.7260
Epoch 11/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5737 - loss: 0.6836 - val_accuracy: 0.4769 - val_loss: 0.7280
Epoch 12/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5737 - loss: 0.6821 - val_accuracy: 0.4821 - val_lo